In [29]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/sephora_clean.csv")
df_clean.head(5)

/var/folders/bx/xxftvqk17n33yx6l437wb7200000gn/T/ipykernel_69004/657677713.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv("../data/processed/sephora_clean.csv")


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,...,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,rating_category
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,i use this with the nudestix citrus clean balm...,Taught me how to double cleanse!,...,0,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,positive
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,i bought this lip mask after reading the revie...,Disappointed,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,negative
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,my review title says it all i get so excited t...,New Favorite Routine,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,ive always loved this formula for a long time ...,Can't go wrong with any of them,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,if you have dry cracked lips this is a must ha...,A must have !!!,...,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,positive


In [30]:
df_nlp = df_clean.copy()

df_nlp["review_title"] = df_nlp["review_title"].fillna("")
df_nlp["review_text"] = df_nlp["review_text"].fillna("")

df_nlp["text"] = df_nlp["review_title"] + " " + df_nlp["review_text"]
df_nlp["text"] = df_nlp["text"].str.strip()

In [31]:
df_nlp = df_nlp[df_nlp["text"].str.len() > 0].copy()

In [32]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)      # linkleri kaldır
    text = re.sub(r"[^a-z\s]", " ", text)            # sadece harfler
    text = re.sub(r"\s+", " ", text).strip()         # fazla boşlukları temizle
    return text

df_nlp["clean_text"] = df_nlp["text"].apply(clean_text)

In [33]:
# 1. MODEL İÇİN GEREKLİ KOLONLARI SEÇ
df_model = df_nlp[[
    "product_id",
    "brand_name",
    "product_name",
    "primary_category",
    "secondary_category",
    "clean_text",
    "rating",
    "price_usd",
    "rating_category"
]].copy()

# sadece positive / negative bırak
df_model = df_model[df_model["rating_category"].isin(["positive", "negative"])].copy()

df_model.head()

,product_id,brand_name,product_name,primary_category,secondary_category,clean_text,rating,price_usd,rating_category
0,P504322,NUDESTIX,Gentle Hydra-Gel Face Cleanser,Skincare,Cleansers,taught me how to double cleanse i use this wit...,5,19.0,positive
1,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,disappointed i bought this lip mask after read...,1,24.0,negative
2,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,new favorite routine my review title says it a...,5,24.0,positive
3,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,can t go wrong with any of them ive always lov...,5,24.0,positive
4,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,a must have if you have dry cracked lips this ...,5,24.0,positive


In [34]:
# 2. SENTIMENT MODELİ KUR
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df_model["clean_text"]
y = df_model["rating_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model', LogisticRegression(max_iter=1000))])

In [35]:
# 3. MODEL PERFORMANSINA BAK
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.89      0.78      0.83     22810
    positive       0.97      0.99      0.98    179389

    accuracy                           0.96    202199
   macro avg       0.93      0.88      0.91    202199
weighted avg       0.96      0.96      0.96    202199



In [36]:
# 4. TÜM VERİYE SENTIMENT TAHMİNİ YAP
df_model["sentiment_pred"] = pipeline.predict(df_model["clean_text"])

In [37]:
# 5. SENTIMENT SCORE ÜRET
# predict_proba çıktısında kolon sırası classes_ ile belirlenir
print(pipeline.named_steps["model"].classes_)

['negative' 'positive']


In [38]:
df_model["sentiment_score"] = pipeline.predict_proba(df_model["clean_text"])[:, 1]

In [39]:
df_model.head(5)

,product_id,brand_name,product_name,primary_category,secondary_category,clean_text,rating,price_usd,rating_category,sentiment_pred,sentiment_score
0,P504322,NUDESTIX,Gentle Hydra-Gel Face Cleanser,Skincare,Cleansers,taught me how to double cleanse i use this wit...,5,19.0,positive,positive,0.983346
1,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,disappointed i bought this lip mask after read...,1,24.0,negative,negative,0.000676
2,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,new favorite routine my review title says it a...,5,24.0,positive,positive,0.997378
3,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,can t go wrong with any of them ive always lov...,5,24.0,positive,positive,0.997508
4,P420652,LANEIGE,Lip Sleeping Mask Intense Hydration with Vitam...,Skincare,Lip Balms & Treatments,a must have if you have dry cracked lips this ...,5,24.0,positive,positive,0.806283


In [41]:
# 6. İSTEDİĞİN SON TABLOYU OLUŞTUR
product_summary = df_model.groupby(
    ["primary_category","secondary_category", "product_id", "brand_name", "product_name"],
    as_index=False
).agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count"),
    avg_sentiment=("sentiment_score", "mean"),
    positive_ratio=("sentiment_pred", lambda x: (x == "positive").mean()),
    negative_ratio=("sentiment_pred", lambda x: (x == "negative").mean()),
    price=("price_usd", "mean")
)

product_summary.head()

,primary_category,secondary_category,product_id,brand_name,product_name,avg_rating,rating_count,avg_sentiment,positive_ratio,negative_ratio,price
0,Skincare,Cleansers,P122651,CLINIQUE,Clarifying Lotion 1,4.594737,190,0.905025,0.931579,0.068421,20.0
1,Skincare,Cleansers,P122661,CLINIQUE,7 Day Face Scrub Cream Rinse-Off Formula,4.596078,765,0.923623,0.939869,0.060131,26.0
2,Skincare,Cleansers,P122718,CLINIQUE,Exfoliating Face Scrub,4.653747,1161,0.935862,0.953488,0.046512,26.0
3,Skincare,Cleansers,P122762,CLINIQUE,Rinse-Off Foaming Cleanser,4.565773,783,0.915367,0.934866,0.065134,23.5
4,Skincare,Cleansers,P122876,CLINIQUE,Clarifying Lotion 4,4.512077,207,0.898423,0.932367,0.067633,20.0


In [42]:
product_summary.to_csv("../data/processed/nlp_product_summary.csv", index=False)